# Task 2 — Built-in Tools + Custom Tools, wired as a LangGraph with Nodes

This version uses **real graph nodes** and LangGraph's built-in
`tools_condition` (no manual classify/extract pipeline).

**Graph shape:**

```
START -> augment_prompt -> assess --(tools_condition)--> tools -> assess -> ... -> END
                                    \-----------------------------------------/ (no tool needed)
```

- `augment_prompt` — enriches the raw human message with the list of
  available tools/instructions before the model sees it.
- `assess` — the LLM node (`llm_with_tools`). It looks at the (augmented)
  conversation and either answers directly or emits a `tool_call`.
- `tools_condition` — LangGraph's built-in router: inspects the last message;
  if it has `tool_calls`, route to the `tools` node, otherwise route to `END`.
- `tools` — a `ToolNode` holding all 5 tools. It reads the tool call's `name`
  and automatically dispatches to the matching tool function — this is where
  "which tool to use" actually gets decided/executed, driven by whatever the
  `assess` node requested.
- Loop `tools -> assess` — feeds the tool's result back to the LLM so it can
  produce a final natural-language answer (or call another tool).

**Contents**
1. Imports & setup
2. 3 built-in LangChain tools
3. Custom tool #1 — Age Calculator
4. Custom tool #2 — Weather API integration
5. Toolkit class
6. Combine tools + bind to the LLM
7. State
8. Nodes: `augment_prompt`, `assess`, `tools`
9. Graph wiring (with `tools_condition`)
10. Compile + test runs


In [ ]:
# ---- Library imports ----
from typing import TypedDict, Annotated, Type
from datetime import datetime, date

from dotenv import load_dotenv

from pydantic import BaseModel, Field

from langchain_core.tools import tool, StructuredTool, BaseTool
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage

from langchain_community.tools import DuckDuckGoSearchRun, ShellTool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Local model via Ollama (supports bind_tools / tool calling)
from langchain_ollama import ChatOllama

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

import requests

In [ ]:
# ---- Environment & LLM setup ----
load_dotenv()

# Requires: `ollama pull phi4-mini` and the Ollama server running locally
llm = ChatOllama(model="phi4-mini", temperature=0)

## 2. Three Built-in Tools

1. `DuckDuckGoSearchRun` — general web search
2. `ShellTool` — run local shell commands
3. `WikipediaQueryRun` — query Wikipedia via `WikipediaAPIWrapper`

In [ ]:
# Built-in tool 1: Web search
search_tool = DuckDuckGoSearchRun(region="us-en")

print(search_tool.name)
print(search_tool.description)

In [ ]:
# Built-in tool 2: Shell access
shell_tool = ShellTool()

print(shell_tool.name)
print(shell_tool.description)

In [ ]:
# Built-in tool 3: Wikipedia lookup
wiki_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
wikipedia_tool = WikipediaQueryRun(api_wrapper=wiki_wrapper)

print(wikipedia_tool.name)
print(wikipedia_tool.description)

## 3. Custom Tool #1 — Age Calculator

Built with the `@tool` decorator (same pattern as `calculator` / `add` /
`multiply` in the uploaded notebooks).

In [ ]:
@tool
def age_calculator(birth_date: str) -> dict:
    """
    Calculate a person's age from their birth date.
    birth_date must be in 'YYYY-MM-DD' format (e.g. '1998-07-23').
    Returns years, months, days, and total days lived.
    """
    try:
        dob = datetime.strptime(birth_date, "%Y-%m-%d").date()
        today = date.today()

        if dob > today:
            return {"error": "Birth date cannot be in the future"}

        years = today.year - dob.year
        months = today.month - dob.month
        days = today.day - dob.day

        if days < 0:
            months -= 1
            prev_month_last_day = (today.replace(day=1) - date.resolution).day
            days += prev_month_last_day

        if months < 0:
            years -= 1
            months += 12

        total_days = (today - dob).days

        return {
            "birth_date": birth_date,
            "years": years,
            "months": months,
            "days": days,
            "total_days_lived": total_days,
        }
    except ValueError:
        return {"error": "birth_date must be in 'YYYY-MM-DD' format"}

In [ ]:
# Manual check
print(age_calculator.name, "=>", age_calculator.description)
print(age_calculator.invoke({"birth_date": "1998-07-23"}))

## 4. Custom Tool #2 — Weather API Integration

Built with `StructuredTool.from_function` + a Pydantic `args_schema` (same
approach as `multiply_tool` in `tools_in_langchain.ipynb`). Calls the free
[Open-Meteo](https://open-meteo.com/) API (geocoding + forecast) — no API key
required.

In [ ]:
class WeatherInput(BaseModel):
    city: str = Field(description="City name to fetch current weather for, e.g. 'Dhaka' or 'London'")


def get_weather_func(city: str) -> dict:
    """Fetch current weather for a given city using the Open-Meteo API (no API key needed)."""
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    geo_res = requests.get(geo_url, params={"name": city, "count": 1}).json()

    if not geo_res.get("results"):
        return {"error": f"Could not find location for '{city}'"}

    location = geo_res["results"][0]
    lat, lon = location["latitude"], location["longitude"]

    weather_url = "https://api.open-meteo.com/v1/forecast"
    weather_res = requests.get(
        weather_url,
        params={"latitude": lat, "longitude": lon, "current_weather": True},
    ).json()

    current = weather_res.get("current_weather", {})

    return {
        "city": location.get("name", city),
        "country": location.get("country"),
        "temperature_C": current.get("temperature"),
        "windspeed_kmh": current.get("windspeed"),
        "weather_code": current.get("weathercode"),
        "observed_at": current.get("time"),
    }


weather_tool = StructuredTool.from_function(
    func=get_weather_func,
    name="get_weather",
    description="Get the current weather (temperature, windspeed) for a given city",
    args_schema=WeatherInput,
)

In [ ]:
# Manual check
print(weather_tool.name, "=>", weather_tool.description)
print(weather_tool.args)

## 5. Toolkit Class

Grouping the two custom tools, following the `MathToolkit` pattern from
`tools_in_langchain.ipynb`.

In [ ]:
class CustomToolkit:
    """Groups the custom tools (age calculator + weather API) for easy reuse."""

    def get_tools(self):
        return [age_calculator, weather_tool]


custom_toolkit = CustomToolkit()

for t in custom_toolkit.get_tools():
    print(t.name, "=>", t.description)

## 6. Combine All Tools & Bind to the LLM

3 built-in tools + 2 custom tools = 5 tools total. `bind_tools` makes the model aware of each tool's name, description, and argument schema, so it can both *select* a tool and *fill in its arguments* in one native tool call.

In [ ]:
# Combine built-in tools + custom toolkit tools
tools = [search_tool, shell_tool, wikipedia_tool] + custom_toolkit.get_tools()

# Make the LLM tool-aware (used inside the "assess" node below)
llm_with_tools = llm.bind_tools(tools)

[t.name for t in tools]

## 7. State

Same `ChatState` shape as `tools.ipynb`: a list of messages that grows via the `add_messages` reducer as the graph runs.

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

## 8. Nodes

### `augment_prompt`
Rewrites the latest human message to include a short tool menu, so the model
has clear context on what's available *before* it decides anything. Because
we reuse the original message's `id`, `add_messages` **updates it in place**
instead of appending a duplicate.

In [ ]:
TOOL_DESCRIPTIONS = {
    "duckduckgo_search": "Search the web for general/current information.",
    "terminal": "Run a local shell command (e.g. list files).",
    "wikipedia": "Look up a factual/encyclopedic topic on Wikipedia.",
    "age_calculator": "Calculate someone's age from a birth date (YYYY-MM-DD).",
    "get_weather": "Get the current weather for a named city.",
}


def augment_prompt_node(state: ChatState):
    """Enrich the latest human message with a tool menu before the model sees it."""
    last_msg = state["messages"][-1]

    tool_menu = "\n".join(f"- {name}: {desc}" for name, desc in TOOL_DESCRIPTIONS.items())
    augmented_content = (
        f"{last_msg.content}\n\n"
        f"(You may use one of these tools if needed:\n{tool_menu})"
    )

    # Same id => add_messages reducer replaces the message instead of appending a new one
    augmented_msg = HumanMessage(content=augmented_content, id=last_msg.id)
    return {"messages": [augmented_msg]}

### `assess`
The LLM node. Reads the (augmented) conversation and either responds directly
or emits a message with `tool_calls` populated.

In [ ]:
def assess_node(state: ChatState):
    """LLM node that may answer directly or request a tool call."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

### `tools`
A single `ToolNode` holding all 5 tools. When `assess` emits a tool call like
`{"name": "get_weather", "args": {"city": "Dhaka"}}`, `ToolNode` matches the
`name` against this list and executes the correct function automatically —
this is where the actual "which tool" dispatch happens.

In [ ]:
tool_node = ToolNode(tools)

## 9. Graph Wiring

```
START -> augment_prompt -> assess --tools_condition--> tools -> assess -> ... -> END
                                                    \-> END (if no tool call)
```

`tools_condition` is LangGraph's built-in router: it looks at the last
message in state; if it has `tool_calls`, it returns `"tools"`, otherwise it
returns `END`. No manual if/else routing needed.

In [ ]:
graph = StateGraph(ChatState)

graph.add_node("augment_prompt", augment_prompt_node)
graph.add_node("assess", assess_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "augment_prompt")
graph.add_edge("augment_prompt", "assess")

# Built-in conditional router: "assess" -> "tools" if a tool call was made, else END
graph.add_conditional_edges("assess", tools_condition)

# Feed the tool result back to the LLM for a final natural-language answer
graph.add_edge("tools", "assess")

In [ ]:
chatbot = graph.compile()

chatbot

## 10. Test Runs

In [ ]:
# No tool needed -> assess routes straight to END
out = chatbot.invoke({"messages": [HumanMessage(content="Hello!")]})
print(out["messages"][-1].content)

In [ ]:
# Custom tool: age calculator
out = chatbot.invoke({
    "messages": [HumanMessage(content="My birth info: I was born on 1998-07-23. How old am I today?")]
})
print(out["messages"][-1].content)

In [ ]:
# Custom tool: weather API
out = chatbot.invoke({
    "messages": [HumanMessage(content="Can you tell me the current weather in Dhaka right now?")]
})
print(out["messages"][-1].content)

In [ ]:
# Built-in tool: Wikipedia
out = chatbot.invoke({
    "messages": [HumanMessage(content="Give me a quick summary of LangChain, the framework.")]
})
print(out["messages"][-1].content)

In [ ]:
# Built-in tool: shell
out = chatbot.invoke({
    "messages": [HumanMessage(content="List the files in the current directory.")]
})
print(out["messages"][-1].content)